In [ ]:
df=pd.read_csv("stable-diffusion\metadata.csv")

In [ ]:
df

In [ ]:
import pandas as pd
import json
import os

# Define paths
dataset_dir = "stable-diffusion\jewellery_dataset"  # Adjust this to your dataset directory
csv_path = "stable-diffusion\metadata.csv"   # Adjust this to your CSV file location

# Ensure the dataset directory exists
os.makedirs(dataset_dir, exist_ok=True)

# Load the CSV
df = pd.read_csv(csv_path)

# Create metadata.jsonl
metadata_path = os.path.join(dataset_dir, "metadata.jsonl")
with open(metadata_path, "w") as f:
    for _, row in df.iterrows():
        file_name = row["image_path"]  # e.g., "bracelets/bracelet1.jpg"
        caption = row["description"]
        json_line = {"file_name": file_name, "text": caption}
        f.write(json.dumps(json_line) + "\n")

In [ ]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.get_device_name(0))  # Prints GPU name

In [ ]:
!git clone https://github.com/huggingface/diffusers.git

In [ ]:
!accelerate launch \
  --num_processes=1 \
  --num_machines=1 \
  --mixed_precision=fp16 \
  --dynamo_backend=no \
  diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="stable-diffusion/jewellery_dataset" \
  --output_dir="output" \
  --mixed_precision="fp16" \
  --resolution=1024 \
  --train_batch_size=1 \
  --learning_rate=1e-5 \
  --max_train_steps=10000 \
  --checkpointing_steps=1000 \
  --validation_prompt="a beautiful necklace" \
  --validation_epochs=1

In [ ]:
import diffusers
print([name for name in dir(diffusers) if "Pipeline" in name])

In [ ]:
import numpy as np
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from diffusers.utils import load_image
from PIL import Image
import torch

# Load the pipeline with the SDXL base model
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
)

# Load fine-tuned LoRA weights from the latest checkpoint
pipe.load_lora_weights("output/checkpoint-9000")

# Optional: Use a better VAE for improved quality
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)
pipe.vae = vae

# Optimize for low GPU memory
pipe.enable_model_cpu_offload()

# Define your test prompts
prompt = "a diamond necklace with intricate details"
negative_prompt = "blurry, low quality"

# Generate the image (without ControlNet for now)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
).images[0]

# Save and display the generated image
image.save("test_generated_image.png")
from IPython.display import display
display(image)

In [ ]:
from diffusers.pipelines import stable_diffusion
print([name for name in dir(stable_diffusion) if "Pipeline" in name])